In [1]:
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
class BoostedTree:
  def __init__(self,X,gradients,hessians,min_child_weight,gamma,reg_lambda,max_depth,ids=None):
    self.X = X
    self.gradients = gradients
    self.hessians = hessians
    self.min_child_weight = min_child_weight
    self.gamma = gamma
    self.reg_lambda = reg_lambda
    self.max_depth = max_depth
    self.req_ids=ids if ids is not None else np.arange(len(gradients))
    self.cnt_feature=X.shape[1]
    self.opt_weight=-1*((self.gradients[self.req_ids].sum())/(self.hessians[self.req_ids].sum()+self.reg_lambda))
    self.threshold=0.0
    self.split_score=0.0
    self.split_ids=0
    self.build_tree()

  def build_tree(self):
    # building the tree recursively after each step finding the best splits, if a valid split is not found, it becomes leaf node
    if self.max_depth<=0:
      return

    for i in range(self.cnt_feature):
      self.find_best_split(i)

    if self.split_score<=0.0:  # no valid split, this will be leaf node
      return

    # partitioning the data based on best split
    X_col=self.X[self.req_ids,self.split_ids]
    left_ids=self.req_ids[X_col<=self.threshold]
    right_ids=self.req_ids[X_col>self.threshold]

    # Building subtrees using recursion
    self.left=BoostedTree(
        self.X,self.gradients,self.hessians,
        self.min_child_weight,self.gamma,self.reg_lambda,
        self.max_depth-1,ids=left_ids
    )

    self.right=BoostedTree(
        self.X,self.gradients,self.hessians,
        self.min_child_weight,self.gamma,self.reg_lambda,
        self.max_depth-1,ids=right_ids
    )


  def find_best_split(self,idx):
    # now let's find the best split for a particular given feature
    X_col=self.X[self.req_ids,idx]
    G=self.gradients[self.req_ids]
    H=self.hessians[self.req_ids]

    #sorting by feature value
    sorted_ids=np.argsort(X_col)
    X_col=X_col[sorted_ids]
    G=G[sorted_ids]
    H=H[sorted_ids]

    G_total=G.sum()
    H_total=H.sum()

    left_G=0.0
    left_H=0.0

    best_gain=-float('inf')
    best_tresh=None
    best_index=None

    for i in range(1,len(X_col)):
      left_G+=G[i-1]
      left_H+=H[i-1]
      right_G=G_total-left_G
      right_H=H_total-left_H

      if X_col[i]==X_col[i-1]:
        continue
      if left_H<self.min_child_weight or right_H<self.min_child_weight:    # it can cause overfitting
        continue

      gain=0.5*((left_G**2)/(left_H+self.reg_lambda)+(right_G**2)/(right_H+self.reg_lambda)-(G_total**2)/(H_total+self.reg_lambda))-self.gamma

      if gain>best_gain:
        best_gain=gain
        best_tresh=(X_col[i]+X_col[i-1])/2.0
        best_index=i

    if best_gain>self.split_score:
      self.split_score=best_gain
      self.threshold=best_tresh
      self.split_ids=idx

  def predict(self,X):
    # predicting for all sample in X
    return np.array([self._predict_single(x) for x in X])

  def _predict_single(self,x):
    # predicting for a single sample
    if self.split_score<=0.0:
      return self.opt_weight

    if x[self.split_ids]<=self.threshold:
      return self.left._predict_single(x)
    else:
      return self.right._predict_single(x)


In [3]:
class XGBoostRegressor:
  def __init__(self,n_estimators=100,learning_rate=0.1,max_depth=3,min_child_weight=1.0,gamma=0.0,reg_lambda=1.0):
    self.n_estimators=n_estimators
    self.learning_rate=learning_rate
    self.max_depth=max_depth
    self.min_child_weight=min_child_weight
    self.gamma=gamma
    self.reg_lambda=reg_lambda
    self.trees=[]

  def fit(self,X,y):
    # intial prediction (y_hat) is zero
    y_pred=np.zeros(len(y))

    for i in range(self.n_estimators):
      # computing gradient and hessian for mse
      grad=y_pred-y  # first derivate of (1/2)(y-y_pred)^2
      hess=np.ones_like(y)  # second derivative is 1

      tree=BoostedTree(
          X,grad,hess,self.min_child_weight,self.gamma,self.reg_lambda,self.max_depth
      )
      # updating predictions
      y_pred+=self.learning_rate*tree.predict(X)
      self.trees.append(tree)

  def predict(self,X):
    # predicting for all samples
    y_pred=np.zeros(len(X))
    for tree in self.trees:
      y_pred+=self.learning_rate*tree.predict(X)
    return y_pred

In [4]:
# Example usage
X=np.array([[1],[2],[3],[4],[5]])
y=np.array([1.1,1.9,3.0,3.9,5.2])

model=XGBoostRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    gamma=0.0,
    reg_lambda=1.0
)
model.fit(X,y)
preds = model.predict(X)
print("Predictions:", preds)

Predictions: [1.09981679 1.89948063 2.99810303 3.89255375 5.16913632]
